# Lesson 4 — Tasks and concurrency

Persistence + concurrency + scheduling + isolation, each ~30 lines, all orthogonal.

Covers `s12` + `s13` + `s14` + `s18`. Narration in `speaker_notes/04_tasks_and_concurrency.md`.

In [ ]:
import json
import subprocess
import threading
import time
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path

from llm_client import get_client, DEFAULT_MODEL

client = get_client()

## Part 1 — File-backed task graph

Each task is a JSON file. `blockedBy` makes it a DAG. `can_start` and `claim_task` are the only two operations that matter.

In [ ]:
TASKS_DIR = Path(".tasks_demo")
TASKS_DIR.mkdir(exist_ok=True)

@dataclass
class Task:
    id: str
    description: str
    status: str = "pending"
    blockedBy: list[str] = field(default_factory=list)
    owner: str | None = None
    worktree: str | None = None

def save_task(t: Task):
    (TASKS_DIR / f"{t.id}.json").write_text(json.dumps(asdict(t), indent=2))

def load_task(task_id: str) -> Task | None:
    p = TASKS_DIR / f"{task_id}.json"
    if not p.exists(): return None
    return Task(**json.loads(p.read_text()))

def list_tasks() -> list[Task]:
    return [Task(**json.loads(p.read_text())) for p in TASKS_DIR.glob("*.json")]

def create_task(description: str, blockedBy=None, worktree=None) -> Task:
    t = Task(id=str(uuid.uuid4())[:8], description=description,
             blockedBy=list(blockedBy or []), worktree=worktree)
    save_task(t)
    return t

def can_start(task_id: str) -> bool:
    t = load_task(task_id)
    if not t: return False
    for dep_id in t.blockedBy:
        dep = load_task(dep_id)
        if dep is None or dep.status != "completed":
            return False
    return True

def claim_task(task_id: str, owner: str) -> str:
    if not can_start(task_id):
        return "blocked"
    t = load_task(task_id)
    if t.status != "pending":
        return f"not pending (status={t.status})"
    t.status = "in_progress"; t.owner = owner
    save_task(t)
    return f"claimed by {owner}"

def complete_task(task_id: str) -> list[str]:
    t = load_task(task_id)
    t.status = "completed"; save_task(t)
    return [u.id for u in list_tasks() if can_start(u.id) and u.status == "pending"]

In [ ]:
# Wipe the demo dir for a clean run
for p in TASKS_DIR.glob("*.json"): p.unlink()

a = create_task("write tests")
b = create_task("refactor module", blockedBy=[a.id])
c = create_task("update docs", blockedBy=[b.id])

print("A can start?", can_start(a.id))
print("B can start?", can_start(b.id))   # False — waits on A
print(claim_task(a.id, "alice"))
print("newly unblocked by completing A:", complete_task(a.id))
print("B can start now?", can_start(b.id))

## Part 2 — Background tasks

Tool returns an ID immediately; daemon thread does the work; results are drained at the top of each agent turn as user messages.

In [ ]:
background_tasks: dict[str, dict] = {}
bg_lock = threading.Lock()

def run_bash(command: str, timeout=60) -> str:
    try:
        out = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout)
        return (out.stdout + out.stderr)[:4000] or "(no output)"
    except Exception as e:
        return f"error: {e}"

def start_background_bash(command: str) -> str:
    bg_id = str(uuid.uuid4())[:8]
    with bg_lock:
        background_tasks[bg_id] = {"status": "running", "cmd": command, "result": None}

    def worker():
        result = run_bash(command, timeout=120)
        with bg_lock:
            background_tasks[bg_id]["status"] = "completed"
            background_tasks[bg_id]["result"] = result

    threading.Thread(target=worker, daemon=True).start()
    return f"background task {bg_id} started; will be notified when done."

def collect_finished_notifications() -> list[dict]:
    """Returns notifications as user messages, then clears them."""
    notes = []
    with bg_lock:
        for bg_id, t in list(background_tasks.items()):
            if t["status"] == "completed":
                notes.append({"role": "user", "content":
                    f"<task_notification id={bg_id} cmd={t['cmd']!r}>\n{t['result']}\n</task_notification>"})
                del background_tasks[bg_id]
    return notes

In [ ]:
# Demo without the LLM — just the harness
bg_id_text = start_background_bash("python -c \"import time; time.sleep(1); print('work finished')\"")
print(bg_id_text)
print("immediately:", collect_finished_notifications())
time.sleep(1.5)
print("after sleep:", collect_finished_notifications())

## Part 3 — Cron scheduler

Scheduler thread populates a queue; the agent drains it at the top of each turn. Same trick as background tasks — external events become user messages.

In [ ]:
scheduled_jobs: dict[str, dict] = {}
cron_queue: list[dict] = []
cron_lock = threading.Lock()
_scheduler_started = False

def schedule_job(prompt: str, delay_seconds: int) -> str:
    job_id = str(uuid.uuid4())[:8]
    with cron_lock:
        scheduled_jobs[job_id] = {
            "id": job_id, "prompt": prompt,
            "next_fire": time.time() + delay_seconds,
        }
    return job_id

def _scheduler_loop():
    while True:
        time.sleep(0.5)
        now = time.time()
        with cron_lock:
            for job_id, job in list(scheduled_jobs.items()):
                if job["next_fire"] <= now:
                    cron_queue.append({"job_id": job_id, "prompt": job["prompt"]})
                    del scheduled_jobs[job_id]   # one-shot in this demo

def start_scheduler():
    global _scheduler_started
    if not _scheduler_started:
        threading.Thread(target=_scheduler_loop, daemon=True).start()
        _scheduler_started = True

def drain_cron_queue() -> list[dict]:
    with cron_lock:
        items, cron_queue[:] = list(cron_queue), []
    return items

In [ ]:
start_scheduler()
schedule_job("check disk usage", delay_seconds=2)
print("queue immediately:", drain_cron_queue())
time.sleep(2.5)
print("queue after wait:", drain_cron_queue())

## Part 4 — Worktree isolation

Two parallel tasks editing the same file collide. `git worktree add` gives each its own working directory + branch backed by the same `.git`. Pass `cwd=worktree_path` to subprocess and they're isolated.

In [ ]:
WORKTREES_DIR = Path(".worktrees_demo")

def create_worktree(name: str) -> str:
    WORKTREES_DIR.mkdir(exist_ok=True)
    path = WORKTREES_DIR / name
    cmd = ["git", "worktree", "add", str(path), "-b", f"wt/{name}", "HEAD"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.stdout + r.stderr

def remove_worktree(name: str) -> str:
    path = WORKTREES_DIR / name
    r = subprocess.run(["git", "worktree", "remove", "--force", str(path)],
                       capture_output=True, text=True)
    # Also delete the branch
    subprocess.run(["git", "branch", "-D", f"wt/{name}"], capture_output=True)
    return r.stdout + r.stderr

def run_bash_in(command: str, cwd: str | None = None) -> str:
    try:
        out = subprocess.run(command, shell=True, capture_output=True, text=True,
                             timeout=30, cwd=cwd)
        return (out.stdout + out.stderr)[:4000] or "(no output)"
    except Exception as e:
        return f"error: {e}"

In [ ]:
# Demo: create a worktree, write a file in it, verify it's isolated.
# (Comment out if you don't want a new branch in your repo.)
import os

name = f"demo-{uuid.uuid4().hex[:6]}"
print(create_worktree(name))

wt_path = WORKTREES_DIR / name
# Use Python for portability (no `pwd`/`ls` on Windows cmd):
(wt_path / "scratch.txt").write_text("HI")
print("in worktree, scratch.txt contains:", (wt_path / "scratch.txt").read_text())
print("in main repo, scratch.txt exists?", Path("scratch.txt").exists())

print(remove_worktree(name))


## Putting it together

All async signals (background results, cron) become user messages at the top of the loop. The loop didn't grow.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "bash", "description": "Run a quick shell command (foreground).",
        "parameters": {"type": "object",
            "properties": {"command": {"type": "string"}}, "required": ["command"]}}},
    {"type": "function", "function": {
        "name": "bash_background",
        "description": "Start a long-running shell command in the background. Returns immediately.",
        "parameters": {"type": "object",
            "properties": {"command": {"type": "string"}}, "required": ["command"]}}},
    {"type": "function", "function": {
        "name": "create_task", "description": "Create a task in the task graph.",
        "parameters": {"type": "object",
            "properties": {"description": {"type": "string"},
                           "blockedBy": {"type": "array", "items": {"type": "string"}}},
            "required": ["description"]}}},
    {"type": "function", "function": {
        "name": "list_tasks", "description": "List all tasks with status.",
        "parameters": {"type": "object", "properties": {}}}},
]

def tool_create_task(description, blockedBy=None):
    t = create_task(description, blockedBy=blockedBy or [])
    return f"created {t.id} — {t.description}"

def tool_list_tasks():
    return "\n".join(f"{t.id} [{t.status}] {t.description} (blocked_by={t.blockedBy})"
                     for t in list_tasks()) or "(no tasks)"

HANDLERS = {
    "bash": run_bash,
    "bash_background": start_background_bash,
    "create_task": tool_create_task,
    "list_tasks": tool_list_tasks,
}

def agent_loop(user_prompt: str, *, max_turns=8):
    messages = [
        {"role": "system", "content":
            "You are an operational agent. Use bash_background for any command "
            "that might take more than a few seconds."},
        {"role": "user", "content": user_prompt},
    ]
    for _ in range(max_turns):
        # Inject async signals as user messages BEFORE the LLM call
        for note in collect_finished_notifications():
            messages.append(note)
        for queued in drain_cron_queue():
            messages.append({"role": "user", "content":
                f"<cron job={queued['job_id']}>\n{queued['prompt']}\n</cron>"})

        resp = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=TOOLS)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if resp.choices[0].finish_reason != "tool_calls":
            return msg.content
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            handler = HANDLERS.get(call.function.name)
            output = handler(**args) if handler else f"unknown: {call.function.name}"
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(output)})
    return "(max_turns reached)"

In [ ]:
# Wipe tasks for clean demo
for p in TASKS_DIR.glob("*.json"): p.unlink()

answer = agent_loop(
    "Plan two tasks: (1) list this directory, (2) count python files — task 2 depends on task 1. "
    "Then list_tasks. Don't actually execute the work — just create them."
)
print("\nfinal:", answer)

## Recap

Tasks + background + cron + worktrees — operational concurrency.

Lesson 5 turns these into a team: inboxes, protocols, autonomous polling, MCP.